# 问题：Middleware 可以叠加使用，那么多个中间件书写顺序重要吗？

非常重要！

比如：
```python
middleware=[
    TrimmerMiddleware(),        # 1. 先修剪消息
    SummarizationMiddleware(),  # 2. 再摘要
    LoggingMiddleware()         # 3. 最后记录日志
]
```

比如：
```python
agent = create_agent(
    model=model,
    tools=[get_weather, get_news],
    middleware=[
        PIIMiddleware(strategy="redact"),                # 1. 最先检查
        ModelCallLimitMiddleware(run_limit=10),          # 2. 限制调用次数
        SummarizationMiddleware(max_tokens_before_summary=500),  # 3. 总结历史
        ToolRetryMiddleware(max_retries=3),              # 4. 重试工具
    ]
)
```

## 补充原理说明
中间件执行遵循**列表从前往后入栈、逆序出栈（洋葱模型）**：
1. 请求下发阶段：按列表**从上到下**依次执行中间件逻辑；
2. 结果返回阶段：按列表**从下到上**倒序回溯执行；
顺序颠倒会直接改变数据处理流程，带来完全不同的运行效果。

In [2]:
from langchain.agents.middleware import AgentMiddleware
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

class Middleware1(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件1] before_model")
        return None

    def after_model(self, state, runtime):
        print("[中间件1] after_model")
        return None


class Middleware2(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件2] before_model")
        return None

    def after_model(self, state, runtime):
        print("[中间件2] after_model")
        return None


class Middleware3(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件3] before_model")
        return None

    def after_model(self, state, runtime):
        print("[中间件3] after_model")
        return None


agent = create_agent(
    model=model,
    tools=[],
    middleware=[Middleware1(), Middleware2(), Middleware3()]
)

print("\n执行一次调用，观察顺序：")
agent.invoke({"messages": [{"role": "user", "content": "测试"}]})

print("\n关键点：")
print(" - before_model：正序执行（1→2→3）")
print(" - after_model：逆序执行（3→2→1）")
print(" - 类似洋葱模型：1→2→3→模型→3→2→1")



执行一次调用，观察顺序：
[中间件1] before_model
[中间件2] before_model
[中间件3] before_model
[中间件3] after_model
[中间件2] after_model
[中间件1] after_model

关键点：
 - before_model：正序执行（1→2→3）
 - after_model：逆序执行（3→2→1）
 - 类似洋葱模型：1→2→3→模型→3→2→1
